[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/00_access.ipynb)

# 00 · Access — getting civic text into units

Multilogue organises analysis around three *aspects* of the Fynesse framework — **Access**, **Assess**, and **Address** — rather than rigid stages. Access is the work of making data electronically usable under a clear contract. This notebook shows how multilogue keeps that contract small, and how named public sources can be switched without changing the rest of the pipeline.

**Why a unit contract?** Downstream methods (extraction, clustering, weighting) should not know whether a paragraph came from a consultation CSV, a Polis export, or a synthetic generator. The reusable method lives above the file format.

In [ ]:
#@title Setup (run first)
# Free path: synthetic units + FakeClient (no API key).
# Optional: load("polis", ...) downloads public open data into a local cache.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from IPython.display import display, HTML, Markdown
from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


## The Unit

A unit is one analysable span: an id, a document id, and text. Optional fields carry context for quote checks and free-form metadata. Multilogue never parses study PDFs here; study packages (or the open-data loaders below) hand it units.

In [ ]:
from multilogue.access import list_sources, load
from multilogue.protocols import Unit

print("Registered sources:", list_sources())
bundle = load("synthetic", n_docs_per_group=2, units_per_doc=2)
units = bundle["units"]
print(f"{len(units)} units; license={bundle['license']!r}")
display(units[0])

## Switching sources (pods-style)

Following the old `pods.datasets.<name>()` pattern, each public corpus is a named loader. The same call shape — `load(name, **kwargs)` — returns units plus citation and licence metadata. Downloads land in a gitignored cache (`MULTILOGUE_DATAPATH` or `~/.cache/multilogue`), never in the repository.

The cell below uses **synthetic** data by default so the notebook runs offline. Uncomment the Polis line to pull a real open CC BY conversation (needs network).

In [ ]:
import pandas as pd

# Default: generated units (no download)
bundle = load("synthetic", n_docs_per_group=3, units_per_doc=3)
# Optional public open data (CC BY 4.0) — uncomment to try:
# bundle = load("polis", conversation="vtaiwan.uberx", max_units=80)

units = bundle["units"]
print(bundle["citation"][:200], "…")
print("path:", bundle["path"])
df = pd.DataFrame(units)
display(df[["unit_id", "doc_id", "text"]].head())
print("docs:", df.doc_id.nunique(), "units:", len(df))

## Diagnostic: units and documents by group

Even at Access time we can check that the switch left us with a usable corpus shape — how many documents per group, how long the texts are — without asking a research question yet (that restraint is Assess).

In [ ]:
from multilogue.assess_base import unit_size_table, docs_per_group
from multilogue import viz

sizes = unit_size_table(units)
docs = docs_per_group(units, "group") if any("group" in u for u in units) else None
display(sizes.describe())
if docs is not None:
    display(docs)
    viz.docs_per_group_bars(docs, path=FIG / "00_docs_per_group.png")
viz.unit_size_hist(sizes, group_col="group", units=units, path=FIG / "00_unit_sizes.png")